In [1]:
import math_toolkit
math_toolkit.notebook.activate()


# rewrite

Change the representation of a SymPy expression when another form is better for algebra, analysis, or export.

`expr.rewrite(target)`  
`expr.rewrite(source, target)`  
`expr.rewrite(target, deep=False)`


In [2]:
sin(x) @Eq@ sin(x).rewrite(exp)


            ⎛ ⅈ⋅x    -ⅈ⋅x⎞ 
         -ⅈ⋅⎝ℯ    - ℯ    ⎠ 
sin(x) = ──────────────────
                 2         

## Core behavior

`rewrite` asks each expression to represent itself using a target form. The result is still an ordinary SymPy expression, and the original expression is unchanged.

The target is usually a SymPy class or function such as `exp`, `gamma`, `factorial`, or `Piecewise`. String targets are useful for custom rewrite rules, such as the `"unhold"` rule used by `NamedFunction`.


## Common patterns

The most common use is changing a representation so another symbolic operation sees the structure you care about.


In [3]:
oscillation = sin(x) + cos(x)
oscillation.rewrite(exp)


    ⎛ ⅈ⋅x    -ⅈ⋅x⎞    ⅈ⋅x    -ⅈ⋅x
  ⅈ⋅⎝ℯ    - ℯ    ⎠   ℯ      ℯ    
- ──────────────── + ──── + ─────
         2            2       2  

In [4]:
exp(I * x).rewrite(cos)


ⅈ⋅sin(x) + cos(x)

Use the two-argument form when only one kind of subexpression should move to the target representation. This is useful when a full rewrite would make the whole formula harder to read.


In [5]:
mixed = sin(x) + cos(x) + tan(x)
mixed.rewrite(sin, exp)


    ⎛ ⅈ⋅x    -ⅈ⋅x⎞                  
  ⅈ⋅⎝ℯ    - ℯ    ⎠                  
- ──────────────── + cos(x) + tan(x)
         2                          

Special-function rewrites are useful when a theorem, simplification, or downstream tool expects a particular family of functions.


In [6]:
factorial_as_gamma = factorial(n).rewrite(gamma)
binomial_as_factorial = binomial(n, k).rewrite(factorial)

md("""Built-in SymPy rewrites can target specific mathematical forms.

- Factorial as gamma: {{ factorial_as_gamma }}
- Binomial as factorials: {{ binomial_as_factorial }}
""")

Built-in SymPy rewrites can target specific mathematical forms.

- Factorial as gamma: $\Gamma\left(n + 1\right)$
- Binomial as factorials: $\frac{n!}{k! \left(- k + n\right)!}$


Rewriting to `Piecewise` can expose assumptions and cases that are hidden inside a compact function call.


In [7]:
r = Symbol("r", real=True)
Abs(r).rewrite(Piecewise)


⎧r   for r ≥ 0
⎨             
⎩-r  otherwise

## Options and Details

**Recursive rewriting.** By default, `rewrite` descends into arguments before rewriting the outer expression. Pass `deep=False` when you want only the outer expression to try the rewrite.

In [8]:
nested = factorial(n + factorial(k))
shallow = nested.rewrite(gamma, deep=False)
deep = nested.rewrite(gamma)

md("""The `deep` option controls whether nested rewrite targets are transformed too.

- Shallow rewrite: {{ shallow }}
- Deep rewrite: {{ deep }}
""")

The `deep` option controls whether nested rewrite targets are transformed too.

- Shallow rewrite: $\Gamma\left(n + k! + 1\right)$
- Deep rewrite: $\Gamma\left(n + \Gamma\left(k + 1\right) + 1\right)$


## Semantics

A rewrite is available only when the expression knows a rule for the requested target. If no rule applies, SymPy usually returns the expression unchanged rather than raising an error.


In [9]:
(x + y).rewrite(gamma)


x + y

## Examples and Applications

### Example: expand authored definitions

`NamedFunction` keeps a symbolic definition opaque until you ask for it. That makes `rewrite("unhold")` a good boundary between readable modeling notation and expanded algebra.

In [10]:
mass = Symbol("mass", positive=True)

@NamedFunction
def Kinetic(p, mass):
    return p**2 / (2 * mass)

@NamedFunction
def Potential(q):
    return q**4 / 4 - q**2 / 2

lagrangian = Kinetic(p, mass) - Potential(q)
lagrangian @Eq@ lagrangian


                                     4    2      2  
                                    q    q      p   
Kinetic(p, mass) - Potential(q) = - ── + ── + ──────
                                    4    2    2⋅mass

After expansion, ordinary SymPy tools see the authored formula instead of an opaque function call.


In [11]:
expanded_lagrangian = lagrangian
diff(expanded_lagrangian, q)


   3    
- q  + q

## Pitfalls

`rewrite` is not the same operation as `simplify`. Rewriting can expose a form where simplification becomes possible, but it does not promise to choose the shortest or prettiest equivalent expression.


In [12]:
tan(x).rewrite(sin)


     2   
2⋅sin (x)
─────────
sin(2⋅x) 

## See also

[Expression](Expression.ipynb)  
[NamedFunction](NamedFunction.ipynb)  
[Function](Function.ipynb)  
[If](If.ipynb)  
[Composing structured expressions](../tutorials/composing_expressions.ipynb)
